In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- Locate repo root (works whether launched from repo root or /notebooks) ---
def _find_root(marker="PROJECT_BRIEF.md"):
    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / marker).exists():
            return cand
    return here

PROJECT  = _find_root()
print(PROJECT)
# --- Paths ---
# SWMM_EXE is machine-specific: set this to YOUR SWMM install (see README).
SWMM_EXE = Path(r"D:\EPA SWMM 5.2.4 (64-bit)\runswmm.exe")
SWMM_DIR = PROJECT / "swmm"
BASE_INP = SWMM_DIR / "base_model.inp"
RUNS_DIR = PROJECT / "runs"
DATA_DIR = PROJECT / "data"
RUNS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# --- Experiment settings ---
N_RUNS = 500
SEED   = 42

# --- Uncertain parameters: name -> (low, high) ---
PARAM_RANGES = {
    "rain_mult":   (0.7, 1.5),      # multiply the 2-yr rainfall timeseries
    "imperv_mult": (0.7, 1.3),      # multiply each subcatchment %Imperv (clip 0-100)
    "surf_n":      (0.011, 0.030),  # impervious overland Manning n (absolute)
    "infil_min":   (0.05, 1.5),     # Horton MinRate, in/hr (absolute)
    "pipe_n":      (0.011, 0.030),  # conduit Manning roughness (absolute)
}
PARAM_NAMES = list(PARAM_RANGES)

assert SWMM_EXE.exists(), f"runswmm.exe not found at {SWMM_EXE}"
assert BASE_INP.exists(), f"base_model.inp not found at {BASE_INP}"
print("Config OK. Base model:", BASE_INP.name, "| runs dir:", RUNS_DIR)

D:\Claude\surrogate_modeling
Config OK. Base model: base_model.inp | runs dir: D:\Claude\surrogate_modeling\runs


In [6]:
from scipy.stats import qmc
sampler = qmc.Halton(d=len(PARAM_RANGES), seed=SEED)
sample = sampler.random(n=N_RUNS)
low, high = [], []
for i in PARAM_RANGES.values():
    low.append(i[0])
    high.append(i[1])
print(low, high)
print(f"Before scale {sample}")
scaled = qmc.scale(sample, low, high)
print(f"After scale {scaled}")

[0.7, 0.7, 0.011, 0.05, 0.011] [1.5, 1.3, 0.03, 1.5, 0.03]
Before scale [[0.55130587 0.15176798 0.14270519 0.30108159 0.0314839 ]
 [0.05130587 0.81843465 0.54270519 0.44393873 0.57693845]
 [0.80130587 0.48510131 0.74270519 0.15822444 0.75875663]
 ...
 [0.01029024 0.52076661 0.66430519 0.41103577 0.76176189]
 [0.76029024 0.29854439 0.26430519 0.55389292 0.21630734]
 [0.26029024 0.96521105 0.86430519 0.26817863 0.94358007]]
After scale [[1.1410447  0.79106079 0.0137114  0.4865683  0.01159819]
 [0.7410447  1.19106079 0.0213114  0.69371116 0.02196183]
 [1.3410447  0.99106079 0.0251114  0.27942545 0.02541638]
 ...
 [0.7082322  1.01245996 0.0236218  0.64600187 0.02547348]
 [1.3082322  0.87912663 0.0160218  0.85314473 0.01510984]
 [0.9082322  1.27912663 0.0274218  0.43885901 0.02892802]]


In [13]:
print(PARAM_NAMES)
df = pd.DataFrame(sample, columns=PARAM_NAMES)
df.head()

['rain_mult', 'imperv_mult', 'surf_n', 'infil_min', 'pipe_n']


,rain_mult,imperv_mult,surf_n,infil_min,pipe_n
0,0.551306,0.151768,0.142705,0.301082,0.031484
1,0.051306,0.818435,0.542705,0.443939,0.576938
2,0.801306,0.485101,0.742705,0.158224,0.758757
3,0.301306,0.262879,0.342705,0.586796,0.213302
4,0.676306,0.929546,0.942705,0.872510,0.940575


In [63]:
with open(r"D:\Claude\surrogate_modeling\swmm\base_model.inp", "r") as f:
    for line in f.readlines():
        print(line.strip())
    


[TITLE]
;;Project Title/Notes

[OPTIONS]
;;Option             Value
FLOW_UNITS           CFS
INFILTRATION         HORTON
FLOW_ROUTING         DYNWAVE
LINK_OFFSETS         DEPTH
MIN_SLOPE            0
ALLOW_PONDING        NO
SKIP_STEADY_STATE    NO

START_DATE           01/01/1998
START_TIME           00:00:00
REPORT_START_DATE    01/01/1998
REPORT_START_TIME    00:00:00
END_DATE             01/01/1998
END_TIME             06:00:00
SWEEP_START          01/01
SWEEP_END            12/31
DRY_DAYS             0
REPORT_STEP          00:01:00
WET_STEP             00:05:00
DRY_STEP             01:00:00
ROUTING_STEP         0:00:15
RULE_STEP            00:00:00

INERTIAL_DAMPING     PARTIAL
NORMAL_FLOW_LIMITED  BOTH
FORCE_MAIN_EQUATION  H-W
VARIABLE_STEP        0.75
LENGTHENING_STEP     0
MIN_SURFAREA         12.566
MAX_TRIALS           8
HEAD_TOLERANCE       0.005
SYS_FLOW_TOL         5
LAT_FLOW_TOL         5
MINIMUM_STEP         0.5
THREADS              1

[EVAPORATION]
;;Data Source    Param

In [77]:
# rain_mult   = params["rain_mult"]
# imperv_mult = params["imperv_mult"]
# surf_n      = params["surf_n"]
# infil_min   = params["infil_min"]
# pipe_n      = params["pipe_n"]

# [CONDUITS]

# ;;Name           From Node        To Node          Length     Roughness  InOffset   OutOffset  InitFlow   MaxFlow   

# ;;-------------- ---------------- ---------------- ---------- ---------- ---------- ---------- ---------- ----------

# C1               J1               J2               185        0.016      0          0          0          0         

# C2               J2               Out1             89         0.016      0          0          0          0 

# [TIMESERIES]

# ;;Name           Date       Time       Value     

# ;;-------------- ---------- ---------- ----------

# 2-yr                        0:00       0.29      

# [SUBCATCHMENTS]

# ;;Name           Rain Gage        Outlet           Area     %Imperv  Width    %Slope   CurbLen  SnowPack        

# ;;-------------- ---------------- ---------------- -------- -------- -------- -------- -------- ----------------

# S1               1                J1               4.55     56.8     1587     2        0                        

# S3               1                J1               3.74     39.5     1456     3.1      0                        

# S2               1                J1               4.74     63       1653     2        0

# [INFILTRATION]

# ;;Subcatchment   Param1     Param2     Param3     Param4     Param5    

# ;;-------------- ---------- ---------- ---------- ---------- ----------

# S1               4.5        0.2        6.5        7          0         

# S3               4.5        0.2        6.5        7          0         

# S2               4.5        0.2        6.5        7          0  

# [SUBAREAS]

# ;;Subcatchment   N-Imperv   N-Perv     S-Imperv   S-Perv     PctZero    RouteTo    PctRouted 

# ;;-------------- ---------- ---------- ---------- ---------- ---------- ---------- ----------

# S1               0.015      0.24       0.06       0.3        25         OUTLET    

# S3               0.015      0.24       0.06       0.3        25         OUTLET    

# S2               0.015      0.24       0.06       0.3        25         OUTLET  
content = []
out = []
with open(r"D:\Claude\surrogate_modeling\swmm\base_model.inp", "r") as f:
    for line in f.readlines():
        stripped = line.strip()
        if stripped.startswith("[") & stripped.endswith("]"):
            current = stripped.upper()
            out.append(line)
            continue
        if line.startswith(";;"):
            out.append(line)
            continue
        if not stripped:
            out.append(line)
            continue
        if current == "[SUBCATCHMENTS]":
            data = line.split()
            data[4] = str(2*float(data[4]))
            out.append((" ").join(data))
            continue
        else:
            out.append(line)
        
    # print("\n".join(out))

In [114]:
lines = ["gggg", "yyyy", "111 Flow Routing Continuity 222", "hhhh", "222 Flow Routing Continuity 333"]
gr = (i for i, ln in enumerate(lines) if "Flow Routing Continuity" in ln)
fr = next((i for i, ln in enumerate(lines) if "Flow Routing Continuity" in ln), None)

In [115]:
print(next(gr,None))
print(next(gr,None))
print(next(gr,None))
print(next(gr,None))

2
4
None
None
